# Import and Read data


In [1]:
import sys
sys.path.append('.')  # Add current directory to path
from training import *
import torch
from itertools import product


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
partition = 100

In [3]:
df = pd.read_csv(f"../../data/top30groups/noGeographic/combined/combined{partition}.csv")

In [4]:
import os 
if not os.path.isdir(f"Results{partition}"):
    os.mkdir(f"Results{partition}")

In [5]:
from pathlib import Path
import numpy as np
from datetime import datetime

def save_metrics_txt(best_metrics, partition, append=False):
    """
    Write best_metrics to results/results_{partition}.txt as key: value lines.
    Set append=True to add another block instead of overwriting.
    """
    results_dir = Path("results")
    results_dir.mkdir(parents=True, exist_ok=True)
    path = results_dir / f"results_{partition}.txt"
    mode = "a" if append else "w"

    # make JSON-friendly scalars for numpy types
    def to_scalar(v):
        if isinstance(v, (np.floating,)):
            return float(v)
        if isinstance(v, (np.integer,)):
            return int(v)
        return v

    with open(path, mode, encoding="utf-8") as f:
        if append:
            f.write("\n" + "="*60 + "\n")
        f.write(f"Run saved: {datetime.now().isoformat(timespec='seconds')}\n")
        # align keys for readability
        width = max(len(k) for k in best_metrics.keys())
        for k in sorted(best_metrics.keys()):
            f.write(f"{k:<{width}} : {to_scalar(best_metrics[k])}\n")

# usage:
# save_metrics_txt(best_metrics, partition="gtd300", append=False)


In [6]:
node_feature_cols = [c for c in df.columns if c!='gname']
edge_feature_cols = ['attacktype1', 'target1', 'nkill']
edge_mode = 'hybrid_equal_knn'
equal_cols = ['attacktype1', 'target1']
k = 10

#{'lr': 0.01, 'n_tree': 100, 'tree_depth': 8, 'tree_feature_rate': 0.3, 'feat_dropout': 0.2, 'out_size_nrf': 1024, 'epochs': 500, 'partition': 'gtd100', 'final_evaluation': False, 'n_class': 30}
import random
param_grid = {
    'h1': [16, 32, 64, 128, 256, 512],
    'h2': [16, 32, 64, 128, 256, 512],
    'dropout': [0.1, 0.3, 0.5],
    'lr': [0.01],
    'n_tree': [50, 80,100],
    'tree_depth': [7,8,9],
    'tree_feature_rate': [0.1,0.3],
    'feat_dropout': [0.1,0.2,0.3],
    'out_size_nrf': [256,512,768,1024]
    }

grid_combos = list(product(*param_grid.values()))
param_names = list(param_grid.keys())
sampled_combos = random.sample(grid_combos, 500)

best_acc = -1
for i,combo in enumerate(sampled_combos):
    print(f"combo {i+1} / {len(sampled_combos)}: {combo}")
    nrf_cfg = {
        **dict(zip(param_names, combo)),
        "epochs": 150,
        "partition": f"gtd{partition}",   # one of: "gtd100", "gtd200", "gtd300", "gtd478"
        "final_evaluation": False,
        "lr": 0.01,
        "n_class": 30
    }

    test_acc, best_epoch, best_metrics, epoch_logs = train_joint_gcn_nrf(
        df,
        node_feature_cols,
        edge_feature_cols,
        edge_mode,
        equal_cols,
        k,
        nrf_cfg,
        weight_decay=5e-4,
        device="cuda"
    )

    if test_acc > best_acc:
        best_acc = test_acc
        best_params = nrf_cfg

print(best_params)


combo 1 / 500: (32, 16, 0.1, 0.01, 50, 7, 0.1, 0.1, 1024)
Best validation acc: 0.4033 @ epoch 121
combo 2 / 500: (256, 16, 0.3, 0.01, 100, 8, 0.1, 0.1, 768)
Best validation acc: 0.3917 @ epoch 149
combo 3 / 500: (256, 32, 0.5, 0.01, 100, 8, 0.3, 0.1, 768)
Best validation acc: 0.4133 @ epoch 120
combo 4 / 500: (256, 16, 0.5, 0.01, 80, 7, 0.1, 0.3, 256)
Best validation acc: 0.3450 @ epoch 141
combo 5 / 500: (256, 512, 0.3, 0.01, 80, 8, 0.3, 0.1, 768)
Best validation acc: 0.4450 @ epoch 143
combo 6 / 500: (512, 64, 0.3, 0.01, 50, 7, 0.1, 0.2, 256)
Best validation acc: 0.4000 @ epoch 149
combo 7 / 500: (64, 32, 0.5, 0.01, 100, 8, 0.1, 0.1, 1024)
Best validation acc: 0.3950 @ epoch 147
combo 8 / 500: (256, 256, 0.5, 0.01, 50, 9, 0.3, 0.3, 256)
Best validation acc: 0.4150 @ epoch 137
combo 9 / 500: (256, 32, 0.1, 0.01, 80, 8, 0.3, 0.3, 512)
Best validation acc: 0.4300 @ epoch 130
combo 10 / 500: (32, 512, 0.3, 0.01, 80, 9, 0.1, 0.1, 256)
Best validation acc: 0.3900 @ epoch 123
combo 11 / 500

In [7]:
save_metrics_txt(best_metrics, partition)